In [1]:
from pyspark.sql.functions import col, current_timestamp

# Catalog is passed as a job parameter (default -> ${var.catalog} per target); the default here
# keeps interactive runs working. Schemas/volumes are the same across dev and prod.
dbutils.widgets.text("catalog", "transport_vic_dev")
CATALOG = dbutils.widgets.get("catalog")

BRONZE_TBL = f"{CATALOG}.`01_bronze`.stops"
SILVER_TBL = f"{CATALOG}.`02_silver`.stops"

df = (spark.read.table(BRONZE_TBL)
        .select(
            col("stop_id"),
            col("stop_name"),
            col("stop_lat").try_cast("double").alias("stop_lat"),
            col("stop_lon").try_cast("double").alias("stop_lon"),
            col("level_id"),
            col("feed_id"),
            col("platform_code"),
            col("_source_file"),
            col("_ingest_ts")
        ).withColumn("_silver_ingest_ts", current_timestamp()
    ).dropDuplicates(["stop_id"])
)

(df.write.format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TBL)
 )


/Users/jacktoke/Codes/Databricks/transport-vic-project/.venv/lib/python3.12/site-packages/databricks/sdk/_widgets/__init__.py:70: UserWarning: 
To use databricks widgets interactively in your notebook, please install databricks sdk using:
	pip install 'databricks-sdk[notebook]'
Falling back to default_value_only implementation for databricks widgets.
  warnings.warn(
